In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy
from ipywidgets import StaticInteract, RangeWidget
from IPython.display import FileLink
from clawpack import riemann

In [ ]:
from clawpack.riemann import riemann_tools

## Advection

In [ ]:
%%bash
make test1_f2py &> f2py_output.txt

In [ ]:
FileLink('f2py_output.txt')

In [ ]:
def rp1_solver_advection(ql,qr,auxl=None,auxr=None,problem_data=None):
    from test_rp1_meqn1 import rp1_driver
    wave,s,amdq,apdq = rp1_driver.call_rp1(1,ql,qr)
    wave = wave.reshape((wave.shape[0],wave.shape[1],1))
    s = s.reshape((len(s),1))
    return wave,s,amdq,apdq

In [ ]:
num_eqn = 1
q_l = 3.  # or array([3.])
q_r = 1.  # or array([1.])
states, s, riemann_eval = riemann_tools.riemann_solution(num_eqn, rp1_solver_advection, q_l, q_r, None) 

## Acoustics

In [ ]:
%%bash
make test2_f2py &> f2py_output.txt

In [ ]:
FileLink('f2py_output.txt')

In [ ]:
def rp1_solver_acoustics(ql,qr,auxl=None,auxr=None,problem_data=None):
    import test_rp1_meqn2
    rho = problem_data['rho']
    bulk = problem_data['bulk']
    test_rp1_meqn2.cparam.rho = rho
    test_rp1_meqn2.cparam.bulk = bulk
    test_rp1_meqn2.cparam.cc = numpy.sqrt(bulk/rho)
    test_rp1_meqn2.cparam.zz = numpy.sqrt(bulk*rho)
    wave,s,amdq,apdq = test_rp1_meqn2.rp1_driver.call_rp1(2,ql,qr)
    wave = wave.reshape((wave.shape[0],wave.shape[1],1))
    s = s.reshape((s.shape[0],1))
    return wave,s,amdq,apdq

In [ ]:
num_eqn = 2
problem_data = {}
problem_data['rho'] = 1.
problem_data['bulk'] = 1.
q_l = numpy.array([2.,0.])
q_r = numpy.array([1.,0.])

states, s, riemann_eval = riemann_tools.riemann_solution(num_eqn, rp1_solver_acoustics, q_l, q_r, problem_data=problem_data) 
riemann_tools.plot_phase(states)

In [ ]:
plot_function = riemann_tools.make_plot_function(states,s,riemann_eval)
StaticInteract(plot_function, t=RangeWidget(0,.9,.1))